# 🎓 Szkolny asystent AI – RAG (komórki kodu z tutoriala)

Uruchamiaj komórki **po kolei** (Shift + Enter). Opisy kroków znajdziesz w pliku `TUTORIAL_uczestnika.md`.

**Przed startem:** włącz GPU (*Środowisko wykonawcze → Zmień typ → T4 GPU*) i dodaj sekret `HF_TOKEN` w zakładce 🔑 Secrets.

### Krok 1 – Instalacja bibliotek i sprawdzenie GPU ⌨️

In [ ]:
!pip install -q transformers accelerate sentence-transformers

import os
import torch
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("GPU:", torch.cuda.get_device_name(0))

### Krok 2 – Ładujemy model językowy ⌨️

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-4-E2B-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("Model załadowany!")

### Krok 3 – Funkcja, która pyta model ⌨️

In [ ]:
def ask_llm(prompt, max_new_tokens=150):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # odcinamy nasze pytanie, zostawiamy samą odpowiedź
    start = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][start:], skip_special_tokens=True).strip()


print(ask_llm("Podaj stolicę Polski jednym zdaniem."))

### Krok 4 – Baza wiedzy, której model NIE zna 📋

In [ ]:
documents = [
    "Regulamin: uczeń, który spóźni się na lekcję więcej niż 15 minut, otrzymuje w dzienniku nieobecność. Spóźnienia do 15 minut są odnotowywane osobno.",
    "Regulamin: nieobecność trzeba usprawiedliwić w ciągu 7 dni od powrotu do szkoły – przez dziennik elektroniczny lub pisemnie (kartka od rodzica).",
    "Regulamin: w czasie lekcji telefon musi być wyłączony i schowany w plecaku. Nauczyciel może pozwolić na jego użycie do konkretnego zadania.",
    "Regulamin: sprawdzian musi być zapowiedziany z tygodniowym wyprzedzeniem. W jednym dniu mogą odbyć się najwyżej dwa sprawdziany.",
    "Regulamin: ocenę ze sprawdzianu można poprawić tylko jeden raz, w ciągu dwóch tygodni od jej otrzymania.",
    "Biblioteka szkolna jest czynna od 7:30 do 15:30. Można wypożyczyć 4 książki na 21 dni i przedłużyć termin jeden raz.",
    "Obiady w stołówce są wydawane od 11:30 do 14:00. Obiad można odwołać najpóźniej do godziny 8:00 w tym samym dniu.",
    "Koło informatyczne spotyka się we wtorki o 15:00 w sali 12. Prowadzi je mgr Anna Bajtowska, a udział jest dobrowolny.",
    "Próbna matura z matematyki odbędzie się w pierwszym tygodniu grudnia i potrwa 180 minut. Wolno używać kalkulatora prostego i tablic matematycznych.",
    "Notatka z biologii: mitochondrium to organellum komórkowe, w którym powstaje ATP w procesie oddychania komórkowego. Nazywa się je „elektrownią komórki”.",
    "Notatka z historii: Mieszko I przyjął chrzest w 966 roku w obrządku łacińskim, za pośrednictwem Czech. Wydarzenie to wprowadziło Polskę do kręgu zachodniego chrześcijaństwa.",
]

print(f"Załadowano {len(documents)} dokumentów.")

### Krok 5 – Eksperyment: model BEZ RAG ⌨️

In [ ]:
pytanie = "Ile mam dni na usprawiedliwienie nieobecności w naszej szkole?"

print("Pytanie:", pytanie)
print("Odpowiedź BEZ RAG:", ask_llm(pytanie))

### 6a. Ładujemy model embeddingów i zamieniamy dokumenty na wektory ⌨️

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

doc_embeddings = embed_model.encode(documents, normalize_embeddings=True)

print(doc_embeddings.shape)

### 6b. Funkcja `search` ⌨️

In [ ]:
def search(query, docs, doc_embs, top_k=1):
    query_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(doc_embs, query_emb.T).flatten()   # podobieństwo do każdego dokumentu
    best = np.argsort(scores)[::-1][:top_k]            # od najlepszego do najgorszego
    return [(float(scores[i]), docs[i]) for i in best]


for score, doc in search("Co grozi za spóźnienie na lekcję?", documents, doc_embeddings, top_k=3):
    print(f"{score:.2f} | {doc[:80]}...")

### Krok 7 – Składamy RAG w całość ⌨️

In [ ]:
def ask_rag(query, docs, doc_embs, top_k=1, min_score=None):
    # 1. WYSZUKAJ
    found = search(query, docs, doc_embs, top_k=top_k)
    for score, doc in found:
        print(f"   🔎 podobieństwo {score:.2f}: {doc[:70]}...")

    if min_score is not None and found[0][0] < min_score:
        return "Nie wiem (nie ma nic podobnego w bazie wiedzy)."

    context = "\n".join(doc for _, doc in found)

    # 2. WKLEJ DO POLECENIA
    prompt = (
        "Odpowiedz na pytanie użytkownika, opierając się TYLKO na podanym kontekście. "
        "Jeśli w kontekście nie ma odpowiedzi, napisz 'Nie wiem'.\n\n"
        f"Kontekst: {context}\n\n"
        f"Pytanie: {query}"
    )

    # 3. WYGENERUJ ODPOWIEDŹ
    return ask_llm(prompt)


pytanie = "Ile mam dni na usprawiedliwienie nieobecności w naszej szkole?"
print("Pytanie:", pytanie)
print("Odpowiedź Z RAG:", ask_rag(pytanie, documents, doc_embeddings))

### Krok 8 – Testujemy asystenta ⌨️

In [ ]:
pytania = [
    "Co się stanie, jeśli spóźnię się na lekcję o pół godziny?",
    "Ile razy mogę poprawiać sprawdzian?",
    "Czy mogę używać telefonu na lekcji?",
    "Do której mogę odwołać obiad?",
    "Ile trwa próbna matura z matematyki?",
    "Jaka będzie jutro pogoda w Bajtowie?",
    "Ile kosztuje obiad w stołówce?",
]

for p in pytania:
    print("❓", p)
    print("💬", ask_rag(p, documents, doc_embeddings))
    print("-" * 60)

### 🧪 Część A, zadanie 1 – przeliczenie embeddingów po dopisaniu dokumentu

In [ ]:
doc_embeddings = embed_model.encode(documents, normalize_embeddings=True)

### B1 – Jak wygląda embedding? ⌨️

In [ ]:
wektor = embed_model.encode("Kot śpi na kanapie.")

print("Długość wektora:", len(wektor))
print("Pierwsze 10 liczb:", np.round(wektor[:10], 3))

### B2 – Podobieństwo zdań ⌨️

In [ ]:
zdania = [
    "Kot śpi na kanapie.",
    "Mruczący kociak drzemie na sofie.",
    "Pies goni piłkę w parku.",
    "Kwantowy procesor działa w temperaturze pokojowej.",
]

emb = embed_model.encode(zdania, normalize_embeddings=True)

for i in range(1, len(zdania)):
    print(f"{float(np.dot(emb[0], emb[i])):5.2f}  ← {zdania[i]}")

### B3 – Szukanie po słowach (Ctrl+F) kontra szukanie po znaczeniu ⌨️

In [ ]:
import re

def slowa(tekst):
    return set(re.findall(r"\w+", tekst.lower()))

def search_keywords(query, docs):
    """Wygrywa dokument z największą liczbą wspólnych słów."""
    wspolne = [len(slowa(query) & slowa(d)) for d in docs]
    best = int(np.argmax(wspolne))
    return docs[best], wspolne[best]


for p in ["Do kiedy rodzic ma wytłumaczyć moją absencję?", "Skąd komórka bierze energię?"]:
    print("❓", p)
    doc, n = search_keywords(p, documents)
    print(f"   📝 po słowach ({n} wspólnych): {doc[:60]}...")
    score, doc = search(p, documents, doc_embeddings)[0]
    print(f"   🧠 po znaczeniu ({score:.2f}):  {doc[:60]}...")

### B4 – Nie każdy model embeddingów jest równie dobry 📋

In [ ]:
testy = [
    ("Co się stanie, jeśli przyjdę na lekcję pół godziny po dzwonku?", 0),
    ("Do kiedy rodzic ma wytłumaczyć moją absencję?", 1),
    ("Czy mogę mieć smartfon na ławce podczas lekcji?", 2),
    ("Ile klasówek może być jednego dnia?", 3),
    ("Dostałem jedynkę ze sprawdzianu, czy mogę ją zmienić?", 4),
    ("Do której czynna jest czytelnia?", 5),
    ("Do której mogę zrezygnować z obiadu?", 6),
    ("Kiedy mogę pójść na zajęcia z programowania?", 7),
    ("Ile trwa egzamin próbny z matematyki?", 8),
    ("Skąd komórka bierze energię?", 9),
    ("Kiedy Mieszko został ochrzczony?", 10),
]

def evaluate(model_name, query_prefix="", doc_prefix=""):
    m = SentenceTransformer(model_name)
    d_embs = m.encode([doc_prefix + d for d in documents], normalize_embeddings=True)
    trafienia = 0
    for pytanie, poprawny in testy:
        q_emb = m.encode([query_prefix + pytanie], normalize_embeddings=True)
        ok = int(np.argmax(np.dot(d_embs, q_emb.T))) == poprawny
        trafienia += ok
        print(f"  {'✅' if ok else '❌'} {pytanie}")
    print(f"→ {model_name}: {trafienia}/{len(testy)}\n")


print("Model ANGIELSKI:")
evaluate("sentence-transformers/all-MiniLM-L6-v2")

print("Model WIELOJĘZYCZNY:")
evaluate("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

### 🧪 B4 – porównanie kolejnych modeli embeddingów (opcjonalnie)

In [ ]:
# Odkomentuj, żeby porównać kolejne modele:
# evaluate("intfloat/multilingual-e5-small", query_prefix="query: ", doc_prefix="passage: ")
# evaluate("BAAI/bge-m3")

### B5 – Mapa znaczeń (dla chętnych) 📋

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

przedmioty = {
    "matematyka": [
        "Pole kwadratu to bok razy bok.", "Rozwiąż równanie kwadratowe za pomocą delty.",
        "Suma kątów w trójkącie wynosi 180 stopni.", "Funkcja liniowa ma wykres w kształcie prostej.",
        "Oblicz pochodną tej funkcji.",
    ],
    "biologia": [
        "Chloroplasty prowadzą fotosyntezę.", "DNA zawiera informację genetyczną.",
        "Serce pompuje krew do całego ciała.", "Mitochondria produkują energię dla komórki.",
        "Żaby są płazami.",
    ],
    "historia": [
        "Bitwa pod Grunwaldem odbyła się w 1410 roku.", "Konstytucja 3 maja została uchwalona w 1791 roku.",
        "Napoleon przegrał pod Waterloo.", "Mieszko I przyjął chrzest w 966 roku.",
        "II wojna światowa zaczęła się w 1939 roku.",
    ],
    "informatyka": [
        "Pętla for powtarza instrukcje wiele razy.", "Zmienna przechowuje wartość w pamięci.",
        "Algorytm sortowania porządkuje listę liczb.", "Błąd w programie nazywamy bugiem.",
        "Sieć neuronowa uczy się na danych.",
    ],
}
kolory = {"matematyka": "tab:blue", "biologia": "tab:green", "historia": "tab:orange", "informatyka": "tab:red"}

wszystkie = [z for lista in przedmioty.values() for z in lista]
etykiety = [k for k, lista in przedmioty.items() for _ in lista]

punkty = PCA(n_components=2, random_state=0).fit_transform(
    embed_model.encode(wszystkie, normalize_embeddings=True)
)

plt.figure(figsize=(9, 6))
for nazwa, kolor in kolory.items():
    idx = [i for i, e in enumerate(etykiety) if e == nazwa]
    plt.scatter(punkty[idx, 0], punkty[idx, 1], c=kolor, s=90, label=nazwa, alpha=0.8)
    for i in idx:
        plt.annotate(wszystkie[i].split()[0], punkty[i], textcoords="offset points", xytext=(6, 4), fontsize=8)
plt.title("Mapa znaczeń zdań (embeddingi spłaszczone do 2D)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### C1 – Dzielenie na fragmenty ⌨️

In [ ]:
def chunk_text(text, sentences_per_chunk=2):
    zdania = re.split(r"(?<=[.!?])\s+", text.strip())
    return [" ".join(zdania[i:i + sentences_per_chunk]) for i in range(0, len(zdania), sentences_per_chunk)]

### C2 – Wklej własny tekst i zadaj pytanie ⌨️

In [ ]:
moj_tekst = """
Wisła jest najdłuższą rzeką w Polsce. Ma długość około 1047 kilometrów. Jej źródła znajdują się w Beskidzie Śląskim.
Wisła przepływa między innymi przez Kraków, Warszawę, Płock, Toruń i Gdańsk. Uchodzi do Morza Bałtyckiego.
Toruń słynie z pierników oraz z tego, że urodził się w nim Mikołaj Kopernik. Starówka Torunia znajduje się na liście UNESCO.
Pierniki toruńskie wypieka się według receptur znanych od średniowiecza. Ich ciasto dojrzewa nawet kilka miesięcy.
"""

chunki = chunk_text(moj_tekst, sentences_per_chunk=2)
chunki_emb = embed_model.encode(chunki, normalize_embeddings=True)

print(f"Tekst podzielony na {len(chunki)} fragmentów.")

pytanie = "Jak długo dojrzewa ciasto na pierniki?"
print("❓", pytanie)
print("💬", ask_rag(pytanie, chunki, chunki_emb, top_k=1))